In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
invoices = pd.read_excel("CRATES-2026\\DIST-DSUPMK\\MARCH\\INITIAL-DATA\\MAR-16.xlsx", sheet_name="Crate Movement Report")
inv = pd.ExcelFile("CRATES-2026\\DIST-DSUPMK\\MARCH\\INITIAL-DATA\\MAR-16.xlsx")

In [18]:
mysheet = inv.sheet_names

# loop through each sheet and print its name
for sheet in mysheet:
    print(sheet)

Crate Movement Report
Sheet1
Sheet3
Sheet4


In [19]:
invoices.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [20]:
invoices.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-03-16,Inv,55,55260113,31
1,2026-03-16,Inv,57,57260105,181
2,2026-03-16,Inv,62,62260101,127
3,2026-03-16,Inv,63,63260085,120
4,2026-03-16,Inv,82,82260090,52


In [21]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        912 non-null    datetime64[ns]
 1   STATUS       912 non-null    object        
 2   A/C          912 non-null    int64         
 3   Invoice No.  912 non-null    int64         
 4   QTY          912 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 35.8+ KB


In [22]:
try:
    credit_notes = pd.read_excel("CRATES-2026\\DIST-DSUPMK\\MARCH\\INITIAL-DATA\\MAR-16.xlsx", sheet_name="Sheet1")
except:
    credit_notes = pd.DataFrame()
    print("Credit note file not found")

In [23]:
credit_notes.columns

Index(['DATES', 'STATUS', 'A/C', 'Invoice No.', 'QTY'], dtype='object')

In [24]:
credit_notes.head()

,DATES,STATUS,A/C,Invoice No.,QTY
0,2026-03-16,CN,62,62260102,1
1,2026-03-16,CN,100,100260084,70
2,2026-03-16,CN,582,582260191,0
3,2026-03-16,CN,704,704260246,1
4,2026-03-16,CN,718,718260183,1


In [25]:
credit_notes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331 entries, 0 to 330
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATES        331 non-null    datetime64[ns]
 1   STATUS       331 non-null    object        
 2   A/C          331 non-null    int64         
 3   Invoice No.  331 non-null    int64         
 4   QTY          331 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 13.1+ KB


In [26]:
credit_notes.tail()

,DATES,STATUS,A/C,Invoice No.,QTY
326,2026-03-16,CN,6375,637526077,3
327,2026-03-16,CN,6376,637626077,2
328,2026-03-16,CN,6377,637726081,6
329,2026-03-16,CN,6378,637826120,2
330,2026-03-16,CN,6379,637926111,3


In [27]:
# Define numeric columns for processing
numeric_columns = ['QTY']

In [28]:
# Clean column names first
credit_notes.columns = credit_notes.columns.str.strip()
invoices.columns = invoices.columns.str.strip()

# Define numeric columns for processing
numeric_columns = ['QTY']

# Ensure QTY is integer and strip whitespace from A/C
invoices['QTY'] = invoices['QTY'].astype(int)
credit_notes['QTY'] = credit_notes['QTY'].astype(int)
invoices['A/C'] = invoices['A/C'].astype(str).str.strip()
credit_notes['A/C'] = credit_notes['A/C'].astype(str).str.strip()

# Only consider credit notes with STATUS == 'CN'
credit_notes_cn = credit_notes[credit_notes['STATUS'] == 'CN']

# Remove invoice rows that have an exact match in credit notes (by A/C and QTY)
filtered_invoices = invoices.copy()
for idx, credit_row in credit_notes_cn.iterrows():
    match = (
        (filtered_invoices['A/C'] == credit_row['A/C']) &
        (filtered_invoices['QTY'] == credit_row['QTY'])
    )
    match_idx = filtered_invoices[match].index
    if not match_idx.empty:
        # Drop only the first matching invoice for this credit note
        filtered_invoices = filtered_invoices.drop(match_idx[0])

# Group remaining invoices by A/C and sum QTY
grouped_df = filtered_invoices.groupby(['A/C'], as_index=False)[numeric_columns].sum()

# Get latest dates from invoices
latest_dates = filtered_invoices.groupby('A/C')['DATES'].max().reset_index()

# Merge latest dates with grouped QTY
final_df = pd.merge(latest_dates, grouped_df, on='A/C', how='left')

print(grouped_df)

      A/C  QTY
0     100   86
1    1000   86
2    1003   49
3    1008   11
4    1009   85
..    ...  ...
763   964   14
764   982   35
765   989   35
766   996    4
767   997   67

[768 rows x 2 columns]


In [29]:
# Export to Excel
final_df.to_excel('CRATES-2026\\DIST-DSUPMK\\MARCH\\FINAL-OUTPUT\\MAR-16-CRATES-INVOICES.xlsx', index=False)

In [30]:
final_df.head()

,A/C,DATES,QTY
0,100,2026-03-16,86
1,1000,2026-03-16,86
2,1003,2026-03-16,49
3,1008,2026-03-16,11
4,1009,2026-03-16,85
